# One 4-bit datapath? — GF4 multi-pass vs FP16 outlier retention (Colab)

**Thesis under test.** The GF4-Engine wants a **single 4-bit datapath — no second FP16 unit**. Weights are
**NVFP4 (E2M1, GPTQ/Hessian-reconstructed)**; activations are **GF4** (Gaussian-optimal FP4) after a
block-Hadamard rotation. The awkward layers are the MLP-output projections (`fc2` / `down_proj`), whose
per-token **activation** outliers collapse single-pass 4-bit — which is why they're normally **kept in FP16**
(that FP16 path *is* the second datapath).

> **Question.** Can those outlier layers instead run **entirely in 4-bit** by giving their activations more
> **residual GF4 passes**, matching the FP16-retention accuracy — so the whole model is one 4-bit datapath?

Every layer is NVFP4; we then sweep the **outlier-layer** activation treatment and compare to FP16 retention:

| profile | bulk acts | outlier acts | datapath |
|---|---|---|---|
| A16 (ref) | FP16 | FP16 | — (acts all 16-bit) |
| **retain** (baseline) | 2-pass GF4 | **FP16** | **two** (4-bit + FP16) |
| 1-datapath all-1p | 1-pass | 1-pass | one (4-bit) |
| 1-datapath out-2p | 2-pass | 2-pass | one |
| 1-datapath out-4p | 2-pass | 4-pass | one |
| 1-datapath out-8p | 2-pass | 8-pass | one |

**Claim holds** if a *1-datapath* row ≈ the *retain* row. Weights are identical across all rows (so this is a
clean read on the activation datapath). `lm_head` stays FP16 in every row (single output layer, not a
per-block datapath). Self-contained — no local CUDA package.

## 1. Setup — Runtime → GPU (A100 40 GB for 13B / outlier-retain memory; L4/A10 24 GB for ~7B).

In [ ]:
!pip -q install "transformers>=4.44" "datasets>=2.20" accelerate scipy
import torch, math, time, gc
print("torch", torch.__version__, "| cuda", torch.cuda.is_available(),
      "|", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")

In [ ]:
try:
    from huggingface_hub import notebook_login
    # notebook_login()   # uncomment once for gated Llama
except Exception as e:
    print("hub login unavailable:", e)

## 2. Config

In [ ]:
CONFIG = dict(
    BLOCK_SIZE = 16, CLIP_RATIO = 2.5, HAD_CAP = 128,
    SEQLEN = 2048, N_SAMPLES = 40,
    OUTLIER = ("fc2", "down_proj"),          # the layers normally kept in FP16 (the 2nd datapath)
    HEAD_FP16 = ("lm_head", "embed_out"),    # always FP16 (single output layer, not a per-block datapath)
    USE_GPTQ = True, N_CALIB = 32, PERCDAMP = 0.01,
    # (bulk activation treatment, outlier activation treatment): "fp16" | "a16" | int pass-count
    PROFILES = {
        "A16 (all acts FP16)":      ("a16",  "fp16"),   # upper reference
        "retain (bulk2 / outFP16)": (2,      "fp16"),   # current method = TWO datapaths
        "1dp all-1pass":            (1,      1),         # naive single 4-bit datapath
        "1dp bulk2 out-2p":         (2,      2),
        "1dp bulk2 out-4p":         (2,      4),
        "1dp bulk2 out-8p":         (2,      8),
    },
)
MODELS = [
    "facebook/opt-1.3b",
    "facebook/opt-2.7b",
    # "facebook/opt-6.7b",
    # "meta-llama/Llama-2-7b-hf",
]
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

## 3. Codebooks, block quantizers, block-Hadamard

In [ ]:
import torch.nn as nn, torch.nn.functional as F
from scipy.linalg import hadamard

GF4_POS  = torch.tensor([0.0, 0.0796082, 0.1737177, 0.2828685,
                         0.3952704, 0.5250730, 0.6961928, 1.0])
E2M1_POS = torch.tensor([0.0, 0.5, 1.0, 1.5, 2.0, 3.0, 4.0, 6.0])

def _block_quant(x, levels_pos, block_size, scale_fn):
    lv = (levels_pos / levels_pos[-1]).to(x.device); shape = x.shape
    x2 = x.reshape(-1, shape[-1]).float(); K = x2.shape[1]
    pad = (block_size - K % block_size) % block_size
    if pad: x2 = F.pad(x2, (0, pad))
    Kp = x2.shape[1]; xb = x2.reshape(-1, block_size)
    scale = scale_fn(xb).clamp_min(1e-8); sign = xb.sign()
    xn = (xb.abs() / scale).clamp(0, 1)
    q = lv[(xn.unsqueeze(-1) - lv.view(1, 1, -1)).abs().argmin(-1)]
    return (sign * scale * q).reshape(-1, Kp)[:, :K].reshape(shape).to(x.dtype)

def quantize_nvfp4_weight(W, block_size):
    return _block_quant(W, E2M1_POS, block_size, lambda xb: xb.abs().amax(-1, keepdim=True))

def gf4_act(x, block_size, clip_ratio):
    return _block_quant(x, GF4_POS, block_size,
                        lambda xb: xb.pow(2).mean(-1, keepdim=True).sqrt() * clip_ratio)

def gf4_npass(x, n_pass, block_size, clip_ratio):
    xf = x.float(); q = torch.zeros_like(xf)
    for _ in range(n_pass):
        q = q + gf4_act(xf - q, block_size, clip_ratio)
    return q.to(x.dtype)

def pick_h(in_features, cap):
    h = 1
    while h * 2 <= min(cap, in_features) and in_features % (h * 2) == 0: h *= 2
    return h
_HCACHE = {}
def had_matrix(h, device):
    key = (h, str(device))
    if key not in _HCACHE:
        _HCACHE[key] = (torch.tensor(hadamard(h), dtype=torch.float32) / math.sqrt(h)).to(device)
    return _HCACHE[key]
def block_had(x, H, h):
    shape = x.shape
    return (x.reshape(-1, shape[-1] // h, h).float() @ H).reshape(shape).to(x.dtype)

## 4. Quantized Linear (per-layer bulk/outlier treatment) + surgery

In [ ]:
# Global activation profile: bulk layers vs outlier (fc2/down_proj) layers.
ACT = {"bulk": "a16", "outlier": "fp16"}

class QLinear(nn.Module):
    def __init__(self, lin, block_size, clip_ratio, had_cap,
                 precomputed_Wq=None, is_outlier=False, keep_fp16=False):
        super().__init__()
        self.out_dtype = lin.weight.dtype; dev = lin.weight.device
        self.is_outlier = is_outlier
        self.h = pick_h(lin.weight.shape[1], had_cap); self.H = had_matrix(self.h, dev)
        if precomputed_Wq is None:
            W = lin.weight.data.float()
            Wr = block_had(W, self.H, self.h) if self.h > 1 else W
            Wq = quantize_nvfp4_weight(Wr, block_size).to(self.out_dtype)
        else:
            Wq = precomputed_Wq.to(self.out_dtype)
        self.register_buffer("Wq", Wq)
        # keep the untouched FP16 weight only for outlier layers (needed by the 'fp16' retain profile)
        self.register_buffer("W16", lin.weight.data.clone() if keep_fp16 else None)
        self.bias = None if lin.bias is None else nn.Parameter(lin.bias.data.clone(), requires_grad=False)
        self.block_size = block_size; self.clip = clip_ratio

    def forward(self, x):
        treat = ACT["outlier"] if self.is_outlier else ACT["bulk"]
        if treat == "fp16":
            W = self.W16 if self.W16 is not None else self.Wq   # full FP16 datapath (no rotation/quant)
            return F.linear(x, W.to(x.dtype), self.bias)
        xr = block_had(x, self.H.to(x.device), self.h) if self.h > 1 else x
        xq = xr if treat == "a16" else gf4_npass(xr, int(treat), self.block_size, self.clip)
        return F.linear(xq.to(self.out_dtype), self.Wq, self.bias)

def _set_child(root, dotted, new):
    parent = root
    for p in dotted.split(".")[:-1]:
        parent = parent[int(p)] if p.isdigit() else getattr(parent, p)
    setattr(parent, dotted.split(".")[-1], new)

def _is_outlier(name, cfg): return any(s in name for s in cfg["OUTLIER"])
def _is_head(name, cfg):    return any(s in name for s in cfg["HEAD_FP16"])

## 5. Data + perplexity

In [ ]:
from datasets import load_dataset
def load_wikitext_ids(tokenizer):
    test = load_dataset("Salesforce/wikitext", "wikitext-2-raw-v1", split="test")
    return tokenizer("\n\n".join(test["text"]), return_tensors="pt").input_ids

@torch.no_grad()
def eval_ppl(model, enc, seqlen, n_samples, device):
    n = min(enc.shape[1] // seqlen, n_samples); nll = cnt = 0
    for i in range(n):
        ids = enc[:, i*seqlen:(i+1)*seqlen].to(device)
        out = model(ids, labels=ids)
        nll += out.loss.float().item() * (seqlen - 1); cnt += seqlen - 1
    return math.exp(nll / cnt)

## 6. Weight quantization: GPTQ/Hessian (block-sequential) — quantizes **every** block layer

Per block: accumulate the per-layer input Hessian $H=\sum_t x_t x_t^\top$ on Hadamard-rotated inputs,
GPTQ-quantize each weight (bulk **and** outlier) to NVFP4 with error feedback through $H^{-1}$, then replay
the quantized block to feed the next. Outlier layers additionally keep their FP16 weight for the retain
baseline. One block's Hessians live at a time → fits 13B. $H=I$ reduces to absmax NVFP4.
`USE_GPTQ=False` uses absmax directly.

In [ ]:
class _StopFwd(Exception): pass
def _find_block_list(model):
    best = None
    for _, mod in model.named_modules():
        if isinstance(mod, nn.ModuleList) and len(mod) >= 2:
            if best is None or len(mod) > len(best): best = mod
    return best

@torch.no_grad()
def gptq_quantize_weight(W, H, block_size, percdamp=0.01, cols_block=128):
    W = W.clone().float(); out_f, in_f = W.shape; H = H.clone().float()
    dead = torch.diag(H) == 0; H[dead, dead] = 1.0; W[:, dead] = 0.0
    ar = torch.arange(in_f, device=W.device)
    H[ar, ar] += percdamp * torch.diag(H).mean().clamp_min(1e-8)
    try:
        Hinv = torch.linalg.cholesky(torch.cholesky_inverse(torch.linalg.cholesky(H)), upper=True)
    except Exception:
        Hinv = torch.eye(in_f, device=W.device)
    lv = (E2M1_POS / E2M1_POS[-1]).to(W.device); Q = torch.zeros_like(W)
    for b0 in range(0, in_f, cols_block):
        b1 = min(b0 + cols_block, in_f); cnt = b1 - b0
        Wb = W[:, b0:b1].clone(); Qb = torch.zeros_like(Wb); Eb = torch.zeros_like(Wb)
        Hb = Hinv[b0:b1, b0:b1]; scale = None
        for i in range(cnt):
            if ((b0 + i) % block_size) == 0:
                g1 = min(i + block_size, cnt)
                scale = Wb[:, i:g1].abs().amax(1).clamp_min(1e-8)
            w = Wb[:, i]; d = Hb[i, i].clamp_min(1e-8)
            xn = (w.abs() / scale).clamp(0, 1)
            q = w.sign() * scale * lv[(xn.unsqueeze(-1) - lv.view(1, -1)).abs().argmin(-1)]
            Qb[:, i] = q; err = (w - q) / d
            Wb[:, i:] -= err.unsqueeze(1) * Hb[i, i:].unsqueeze(0); Eb[:, i] = err
        Q[:, b0:b1] = Qb
        if b1 < in_f: W[:, b1:] -= Eb @ Hinv[b0:b1, b1:]
    return Q

@torch.no_grad()
def quantize_weights(model, tokenizer, cfg, device):
    if not cfg["USE_GPTQ"]:                                   # absmax fallback
        nq = 0
        for nm, m in [(n, mm) for n, mm in model.named_modules() if isinstance(mm, nn.Linear)]:
            if _is_head(nm, cfg): continue
            _set_child(model, nm, QLinear(m, cfg["BLOCK_SIZE"], cfg["CLIP_RATIO"], cfg["HAD_CAP"],
                                          is_outlier=_is_outlier(nm, cfg), keep_fp16=_is_outlier(nm, cfg)))
            nq += 1
        print(f"  [absmax] quantized {nq} block linears"); return model

    blocks = _find_block_list(model); assert blocks is not None
    model.config.use_cache = False
    enc = load_wikitext_ids(tokenizer); sl = cfg["SEQLEN"]
    n = min(enc.shape[1] // sl, cfg["N_CALIB"]); samples = [enc[:, i*sl:(i+1)*sl] for i in range(n)]
    cap = {"inps": [], "args": [], "kw": []}; orig0 = blocks[0]
    class Catcher(nn.Module):
        def __init__(self, m): super().__init__(); self.m = m
        def forward(self, hs, *a, **kw):
            cap["inps"].append(hs.detach()); cap["args"].append(a); cap["kw"].append(kw); raise _StopFwd
    blocks[0] = Catcher(orig0)
    for s in samples:
        try: model(s.to(device))
        except _StopFwd: pass
    blocks[0] = orig0
    inps, args, kw = cap["inps"], cap["args"], cap["kw"]

    for li in range(len(blocks)):
        blk = blocks[li]
        tgts = {nm: m for nm, m in blk.named_modules() if isinstance(m, nn.Linear)}  # ALL (bulk + outlier)
        st, hooks = {}, []
        for nm, m in tgts.items():
            in_f = m.weight.shape[1]; h = pick_h(in_f, cfg["HAD_CAP"])
            st[nm] = {"H": torch.zeros(in_f, in_f, device=device), "h": h, "n": 0}
            def mk(key):
                def hook(mod, inp):
                    x = inp[0].reshape(-1, inp[0].shape[-1]); hh = st[key]["h"]
                    xr = (block_had(x, had_matrix(hh, device), hh) if hh > 1 else x).float()
                    st[key]["H"] += xr.t() @ xr; st[key]["n"] += xr.shape[0]
                return hook
            hooks.append(m.register_forward_pre_hook(mk(nm)))
        for j in range(len(inps)): blk(inps[j].to(device), *args[j], **kw[j])
        for hk in hooks: hk.remove()
        for nm, m in list(tgts.items()):
            h = st[nm]["h"]; H = st[nm]["H"] / max(st[nm]["n"], 1)
            W = m.weight.data.float()
            Wr = block_had(W, had_matrix(h, device), h) if h > 1 else W
            Wq = gptq_quantize_weight(Wr, H, cfg["BLOCK_SIZE"], cfg["PERCDAMP"]).to(m.weight.dtype)
            out = _is_outlier(nm, cfg)
            _set_child(blk, nm, QLinear(m, cfg["BLOCK_SIZE"], cfg["CLIP_RATIO"], cfg["HAD_CAP"],
                                        precomputed_Wq=Wq, is_outlier=out, keep_fp16=out))
            del H
        st.clear(); gc.collect(); torch.cuda.empty_cache()
        ob, oo = ACT["bulk"], ACT["outlier"]; ACT["bulk"], ACT["outlier"] = "a16", "fp16"
        outs = []
        for j in range(len(inps)):
            o = blk(inps[j].to(device), *args[j], **kw[j])
            outs.append((o[0] if isinstance(o, tuple) else o).detach())
        ACT["bulk"], ACT["outlier"] = ob, oo; inps = outs
        print(f"  [GPTQ] block {li+1}/{len(blocks)}", end="\r")
    print(); return model

## 7. Run the sweep

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer
import pandas as pd

rows = []
for model_id in MODELS:
    print(f"\n=== {model_id} ===  (GPTQ={CONFIG['USE_GPTQ']})")
    t0 = time.time()
    tok = AutoTokenizer.from_pretrained(model_id, use_fast=False)
    try:
        model = AutoModelForCausalLM.from_pretrained(
            model_id, torch_dtype=torch.float16, low_cpu_mem_usage=True).to(DEVICE).eval()
    except torch.cuda.OutOfMemoryError:
        print("  OOM loading — skip."); torch.cuda.empty_cache(); continue
    model.config.use_cache = False
    enc = load_wikitext_ids(tok)
    res = {"model": model_id, "fp16": eval_ppl(model, enc, CONFIG["SEQLEN"], CONFIG["N_SAMPLES"], DEVICE)}
    print(f"  FP16 (unquantized): {res['fp16']:.3f}")

    quantize_weights(model, tok, CONFIG, DEVICE)          # every block layer -> NVFP4

    base = None
    for pname, (bt, ot) in CONFIG["PROFILES"].items():
        ACT["bulk"], ACT["outlier"] = bt, ot
        v = eval_ppl(model, enc, CONFIG["SEQLEN"], CONFIG["N_SAMPLES"], DEVICE)
        res[pname] = v
        if "retain" in pname: base = v
        tag = "" if base is None else f"  (dP vs retain {v - base:+.3f})"
        print(f"  {pname:26s} {v:8.3f}{tag}")
    res["minutes"] = round((time.time() - t0) / 60, 1)
    rows.append(res); pd.DataFrame(rows).to_csv("gf4_datapath_results.csv", index=False)
    del model; gc.collect(); torch.cuda.empty_cache()

df = pd.DataFrame(rows); df

## 8. Plot — does a single 4-bit datapath reach the FP16-retention baseline?

In [ ]:
import matplotlib.pyplot as plt
if len(rows):
    pnames = list(CONFIG["PROFILES"].keys())
    fig, ax = plt.subplots(figsize=(1.9*len(rows)+4, 4.4))
    labels = [m.split("/")[-1] for m in df["model"]]; x = range(len(df)); w = 0.8/len(pnames)
    for j, p in enumerate(pnames):
        ax.bar([i + (j - len(pnames)/2)*w + w/2 for i in x], df[p], w, label=p)
    ax.set_xticks(list(x)); ax.set_xticklabels(labels, rotation=15, ha="right")
    ax.set_ylabel("WikiText-2 PPL"); ax.set_title("Single 4-bit datapath vs FP16 outlier retention")
    ax.legend(fontsize=8); plt.tight_layout(); plt.savefig("gf4_datapath.png", dpi=140); plt.show()
    print("\nΔ vs 'retain' baseline (near-zero = single 4-bit datapath matches FP16 retention):")
    for _, r in df.iterrows():
        base = next(r[p] for p in pnames if "retain" in p)
        s = "  ".join(f"{p.split()[-1]}:{r[p]-base:+.2f}" for p in pnames if p.startswith("1dp"))
        print(f"  {r['model'].split('/')[-1]:<20} retain={base:.3f} | {s}")

## Notes

- **The result to look for.** The smallest outlier pass count whose **1dp** row ≈ the **retain** row is your
  "no second datapath" number: that many GF4 passes on `fc2`/`down_proj` replaces their FP16 unit. Expect
  the required count to **grow with model size** (bigger outliers) and the naive `all-1pass` row to collapse.
- **Weights identical across rows** (all NVFP4/GPTQ), so any gap is purely the outlier *activation* datapath.
- **Cost bookkeeping.** A single 4-bit datapath that runs the ~2 outlier layers/block at k passes costs k×
  their activation-compute — cheaper than an FP16 unit's area if k is small. Report the k you find.
- **Memory.** Outlier layers keep an FP16 weight copy for the retain baseline; for 13B on 24 GB, drop the
  `A16`/`retain` rows (or move `W16` to CPU) if you only need the single-datapath numbers.
- **`lm_head`** stays FP16 in all rows (one output layer, not a per-block datapath). Add it to `OUTLIER` to
  fold it in. Swap `GF4_POS` for learned levels to test codebook variants.
